In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

print("Libraries Loaded")

Libraries Loaded


In [6]:
# ==========================================
# LOAD CLEANED DATA (SAFE)
# ==========================================

df = pd.read_csv("Loan_Default_CLEANED.csv")

print("Dataset Loaded")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Dataset Loaded
Rows: 128345
Columns: 51


,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,income_category,credit_category,risk_score,risk_category,customer_segment,equity_position,equity_pct,Status Label,loan_to_income_ratio,debt_service_coverage
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,Low,Excellent,15,Low Risk,Needs Monitoring,1500.0,1.271186,Default,5.579502,3.584549
1,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,Upper-Mid,Excellent,0,Low Risk,Needs Monitoring,101500.0,19.980315,Repaid,3.573312,5.597048
2,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,High,Fair,20,Low Risk,Standard,201500.0,30.623100,Repaid,3.202160,6.245783
3,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,High,Fair,5,Low Risk,Needs Monitoring,61500.0,8.113456,Repaid,5.559547,3.597416
4,24895,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,High,Excellent,0,Low Risk,Premium,301500.0,29.910714,Repaid,5.840774,3.424204


In [10]:
# =====================================================
# STEP 2 : SELECT ML FEATURES
# =====================================================

features = [

    'loan_amount',
    'income',
    'Credit_Score',
    'LTV',
    'dtir1',
    'rate_of_interest',
    'term',
    'property_value',
    'age_numeric',

    'loan_type',
    'Region',
    'approv_in_adv',
    'business_or_commercial'

]

X = df[features].copy()

y = df['Status'].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("Default Rate:", y.mean()*100)

Features shape: (128345, 13)
Target shape: (128345,)
Default Rate: 18.492344851766724


In [12]:
# =====================================================
# STEP 3 : ENCODE CATEGORICAL DATA
# =====================================================

from sklearn.preprocessing import LabelEncoder

X_encoded = X.copy()

categorical_cols = [

    'loan_type',
    'Region',
    'approv_in_adv',
    'business_or_commercial'

]

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))

    encoders[col] = le


print("Encoding Done")

X_encoded.head()

Encoding Done


,loan_amount,income,Credit_Score,LTV,dtir1,rate_of_interest,term,property_value,age_numeric,loan_type,Region,approv_in_adv,business_or_commercial
0,116500,1740.0,758,98.728814,45.0,3.99,360.0,118000.0,30,0,3,0,1
1,406500,9480.0,834,80.019685,46.0,4.56,360.0,508000.0,40,0,3,1,1
2,456500,11880.0,587,69.376900,42.0,4.25,360.0,658000.0,50,0,0,0,1
3,696500,10440.0,602,91.886544,39.0,4.00,360.0,758000.0,30,0,0,1,1
4,706500,10080.0,864,70.089286,40.0,3.99,360.0,1008000.0,40,0,0,1,1


In [13]:
# =====================================================
# STEP 4 : TRAIN TEST SPLIT
# =====================================================

from sklearn.model_selection import train_test_split


X_train,X_test,y_train,y_test = train_test_split(

    X_encoded,
    y,

    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Train Size:",X_train.shape)
print("Test Size:",X_test.shape)

Train Size: (102676, 13)
Test Size: (25669, 13)


In [18]:
# =====================================================
# STEP 5 : LOGISTIC REGRESSION
# =====================================================

from sklearn.linear_model import LogisticRegression


lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train,y_train)

lr_pred = lr_model.predict(X_test)

lr_proba = lr_model.predict_proba(X_test)[:,1]


print("Logistic Regression Model Ready")

Logistic Regression Model Ready


C:\Users\Admin\anaconda3\anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [17]:
# =====================================================
# FIX : REMOVE ANY REMAINING MISSING VALUES
# =====================================================

print("Missing values check:")

print(X_train.isnull().sum())

print("\nTotal Missing Train:",X_train.isnull().sum().sum())

print("\nTotal Missing Test:",X_test.isnull().sum().sum())


# Fill missing with median

X_train = X_train.fillna(X_train.median())

X_test = X_test.fillna(X_train.median())


print("\nMissing Values Fixed")

Missing values check:
loan_amount                0
income                     0
Credit_Score               0
LTV                        0
dtir1                      0
rate_of_interest           0
term                      31
property_value             0
age_numeric                0
loan_type                  0
Region                     0
approv_in_adv              0
business_or_commercial     0
dtype: int64

Total Missing Train: 31

Total Missing Test: 8

Missing Values Fixed


In [19]:
# =====================================================
# STEP 6 : RANDOM FOREST
# =====================================================

from sklearn.ensemble import RandomForestClassifier


rf_model = RandomForestClassifier(

    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train,y_train)

rf_pred = rf_model.predict(X_test)

rf_proba = rf_model.predict_proba(X_test)[:,1]


print("Random Forest Model Ready")

Random Forest Model Ready


In [20]:
# =====================================================
# STEP 7 : MODEL EVALUATION
# =====================================================

from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score


print("--------------- Logistic Regression ---------------")

print("Accuracy:",accuracy_score(y_test,lr_pred))

print("Precision:",precision_score(y_test,lr_pred))

print("Recall:",recall_score(y_test,lr_pred))

print("F1 Score:",f1_score(y_test,lr_pred))

print("ROC AUC:",roc_auc_score(y_test,lr_proba))


print("\n")

print("--------------- Random Forest ---------------")

print("Accuracy:",accuracy_score(y_test,rf_pred))

print("Precision:",precision_score(y_test,rf_pred))

print("Recall:",recall_score(y_test,rf_pred))

print("F1 Score:",f1_score(y_test,rf_pred))

print("ROC AUC:",roc_auc_score(y_test,rf_proba))

--------------- Logistic Regression ---------------
Accuracy: 0.8354045736101913
Precision: 0.8826979472140762
Recall: 0.1268169370128502
F1 Score: 0.2217719653711549
ROC AUC: 0.6382842794649288


--------------- Random Forest ---------------
Accuracy: 0.9164361681405586
Precision: 0.7807509710832974
Recall: 0.7621655782599537
F1 Score: 0.7713463383434602
ROC AUC: 0.9716409975785147
